# Task [A2]: Predicting Concrete Compressive Strength
**AI-101 Introduction to Artificial Intelligence • Fall 2026**  
**Department of Intelligent Systems • The University of Lahore**  
**Student Name:** Muhammad Zaryab Attiq | **Roll No.:** 70176119 | **Section:** Section A  
**Repository:** https://github.com/Muhammad-Zaryab-Attiq/AI101-ML-Assignment-70176119  
**Instructor:** Dr. Muhammad Awais, AI Architect


## 1. Introduction
Testing the compressive strength of concrete via traditional hydraulic crushing cylinders requires up to 28 days of curing in water baths, creating severe delays in modern construction timelines and quality assurance workflows. In this investigation, we formulate a supervised regression problem to estimate the 28-day compressive strength (in MegaPascals, MPa) directly from raw constituent mix proportions and curing age. By augmenting the feature space with physical chemistry formulations—namely $\log(\text{Age})$ to linearize hydration kinetics and the water-to-cement ratio following Abrams' Law—we demonstrate that gradient boosting regression achieves a test-set RMSE of 4.555 MPa and an $R^2$ of 0.919, outperforming the un-engineered linear regression baseline by over 53% in error reduction.


## 2. Problem Statement
Given an input vector of mix design components and sample maturity $x \in \mathbb{R}^d$ ($d=8$ for raw constituents, $d=10$ with engineered physical ratios), our objective is to learn a hypothesis function $f: \mathbb{R}^d \to \mathbb{R}$ that predicts the concrete compressive strength $y \in [2.33, 82.60]\text{ MPa}$. Success is evaluated using Root Mean Squared Error (RMSE) as the primary penalization metric for large safety-critical deviations, supplemented by Mean Absolute Error (MAE) and the coefficient of determination ($R^2$). The central physical constraint is the pronounced non-linearity of cementitious hydration, wherein strength increases logarithmically during early curing before plateauing, severely violating standard Gauss-Markov linearity assumptions.


## 3. Dataset Used
We utilize the Concrete Compressive Strength dataset donated by Prof. I-Cheng Yeh to the UCI Machine Learning Repository (Dataset ID: 165). The dataset comprises 1,030 experimental concrete formulations. Each sample consists of seven constituent ingredient masses measured in $\text{kg/m}^3$ (Cement, Blast Furnace Slag, Fly Ash, Water, Superplasticizer, Coarse Aggregate, Fine Aggregate) and the specimen curing age in days (1 to 365). Inspection reveals zero missing values and zero duplicate records across all 8 input columns. All numerical inputs were scaled using training-set statistics within a scikit-learn Pipeline to guarantee zero data leakage.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from ucimlrepo import fetch_ucirepo

# Set global reproducibility seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Fetch UCI Dataset
concrete = fetch_ucirepo(id=165)
X_raw = concrete.data.features.copy()
y_raw = concrete.data.targets.copy()

X_raw.columns = ['cement', 'blast_furnace_slag', 'fly_ash', 'water',
                 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age']
y_raw.columns = ['strength']
df = pd.concat([X_raw, y_raw], axis=1)
print(f"Loaded {len(df)} samples with {X_raw.shape[1]} input features.")
print(f"Missing values across all features: {df.isnull().sum().sum()}")


Loaded 1030 samples with 8 input features.
Missing values across all features: 0


## 4. Dataset Statistics & Exploratory Data Analysis (EDA)
The target compressive strength spans from 2.33 MPa to 82.60 MPa with a mean of $35.82 \pm 16.71\text{ MPa}$. The curing age is heavily right-skewed with a median of 28 days and a maximum of 365 days.


In [7]:
# Display summary statistics
stats_summary = df.describe().T[['mean', 'std', 'min', '50%', 'max']]
print(stats_summary.round(2))

# Compute Pearson correlation with target
corr_with_target = df.corr()['strength'].sort_values(ascending=False)
print("\nPearson Correlation with Compressive Strength:")
print(corr_with_target.round(3))


mean     std     min     50%     max
cement            281.17  104.51  102.00  272.90  540.00
blast_furnace_slag 73.90   86.28    0.00   22.00  359.40
fly_ash            54.19   63.99    0.00    0.00  200.10
water             181.57   21.35  121.80  185.00  247.00
superplasticizer    6.20    5.97    0.00    6.40   32.20
coarse_aggregate  972.92   77.75  801.00  968.00 1145.00
fine_aggregate    773.58   80.18  594.00  779.50  992.60
age                45.66   63.17    1.00   28.00  365.00
strength           35.82   16.71    2.33   34.44   82.60

Pearson Correlation with Compressive Strength:
strength              1.000
cement                0.498
superplasticizer      0.366
age                   0.329
blast_furnace_slag    0.135
fly_ash              -0.106
coarse_aggregate     -0.165
fine_aggregate       -0.167
water                -0.290


## 5. Methodology & Zero-Leakage Pipeline

### 5.1 Learning Setting
We adopt a supervised regression framework. Four model families are benchmarked:
1. **Ordinary Least Squares Linear Regression:** Linear parametric baseline.
2. **Polynomial Features + Ridge Regularization:** Second-order polynomial expansion with $L_2$ shrinkage to capture pairwise chemical interactions.
3. **Random Forest & Gradient Tree Boosting:** Non-parametric ensemble methods capable of capturing arbitrary step-wise non-linear response surfaces.
4. **Multi-Layer Perceptron (MLPRegressor):** Feedforward neural network with ReLU activations and Adam optimizer.

### 5.2 Data Split Approach
The dataset of 1,030 samples is partitioned into an 80% training set (824 instances) and a 20% held-out test set (206 instances) using a fixed random seed (`seed = 42`). All feature standardization (`StandardScaler`) is enclosed within scikit-learn `Pipeline` objects, ensuring mean and variance are computed strictly on the training fold without data leakage into validation or testing sets. Hyperparameters are tuned via 5-fold cross-validation exclusively on the training split.


In [9]:
# Feature Engineering: Adding Water/Cement Ratio and log(Age)
df['water_cement_ratio'] = df['water'] / df['cement']
df['log_age'] = np.log(df['age'])

X_raw_feat = df[['cement', 'blast_furnace_slag', 'fly_ash', 'water',
                 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age']]
X_eng_feat = df[['cement', 'blast_furnace_slag', 'fly_ash', 'water',
                 'superplasticizer', 'coarse_aggregate', 'fine_aggregate', 'age',
                 'water_cement_ratio', 'log_age']]
y = df['strength']

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(X_raw_feat, y, test_size=0.20, random_state=RANDOM_SEED)
X_tr_eng, X_te_eng, _, _ = train_test_split(X_eng_feat, y, test_size=0.20, random_state=RANDOM_SEED)

print(f"Training split: {len(X_tr_raw)} samples | Test split: {len(X_te_raw)} samples")


Training split: 824 samples | Test split: 206 samples


## 6. Model Training, 5-Fold CV Tuning & Evaluation
We train each model family across both the raw (8-feature) and engineered (10-feature) representations.


In [11]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Baseline Linear Regression
pipe_lr_raw = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
pipe_lr_eng = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])

cv_lr_raw = np.sqrt(-cross_val_score(pipe_lr_raw, X_tr_raw, y_train, cv=cv, scoring='neg_mean_squared_error')).mean()
cv_lr_eng = np.sqrt(-cross_val_score(pipe_lr_eng, X_tr_eng, y_train, cv=cv, scoring='neg_mean_squared_error')).mean()

pipe_lr_raw.fit(X_tr_raw, y_train)
pipe_lr_eng.fit(X_tr_eng, y_train)

y_pred_lr_raw = pipe_lr_raw.predict(X_te_raw)
y_pred_lr_eng = pipe_lr_eng.predict(X_te_eng)

print(f"Linear Regression (Raw):        CV RMSE = {cv_lr_raw:.3f} | Test RMSE = {np.sqrt(mean_squared_error(y_test, y_pred_lr_raw)):.3f} | Test R2 = {r2_score(y_test, y_pred_lr_raw):.3f}")
print(f"Linear Regression (Engineered):  CV RMSE = {cv_lr_eng:.3f} | Test RMSE = {np.sqrt(mean_squared_error(y_test, y_pred_lr_eng)):.3f} | Test R2 = {r2_score(y_test, y_pred_lr_eng):.3f}")


Linear Regression (Raw):        CV RMSE = 10.627 | Test RMSE = 9.796 | Test R2 = 0.628
Linear Regression (Engineered):  CV RMSE = 7.060 | Test RMSE = 6.562 | Test R2 = 0.833


In [12]:
# Tuned Polynomial + Ridge, Random Forest, Gradient Boosting, MLPRegressor
grid_poly = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('poly', PolynomialFeatures(degree=2, include_bias=False)), ('ridge', Ridge(random_state=RANDOM_SEED))]),
    {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}, cv=cv, scoring='neg_mean_squared_error'
).fit(X_tr_eng, y_train)

grid_rf = GridSearchCV(
    Pipeline([('rf', RandomForestRegressor(random_state=RANDOM_SEED))]),
    {'rf__n_estimators': [100, 200], 'rf__max_depth': [10, 15]}, cv=cv, scoring='neg_mean_squared_error'
).fit(X_tr_eng, y_train)

grid_gbt = GridSearchCV(
    Pipeline([('gbt', GradientBoostingRegressor(random_state=RANDOM_SEED))]),
    {'gbt__n_estimators': [100, 200], 'gbt__learning_rate': [0.05, 0.1], 'gbt__max_depth': [3, 5]}, cv=cv, scoring='neg_mean_squared_error'
).fit(X_tr_eng, y_train)

champion = grid_gbt.best_estimator_
y_pred_champ = champion.predict(X_te_eng)

print(f"Best Poly Ridge:     CV RMSE = {np.sqrt(-grid_poly.best_score_):.3f} | Test RMSE = {np.sqrt(mean_squared_error(y_test, grid_poly.predict(X_te_eng))):.3f} | R2 = {r2_score(y_test, grid_poly.predict(X_te_eng)):.3f}")
print(f"Best Random Forest:  CV RMSE = {np.sqrt(-grid_rf.best_score_):.3f} | Test RMSE = {np.sqrt(mean_squared_error(y_test, grid_rf.predict(X_te_eng))):.3f} | R2 = {r2_score(y_test, grid_rf.predict(X_te_eng)):.3f}")
print(f"Champion GBDT:       CV RMSE = {np.sqrt(-grid_gbt.best_score_):.3f} | Test RMSE = {np.sqrt(mean_squared_error(y_test, y_pred_champ)):.3f} | R2 = {r2_score(y_test, y_pred_champ):.3f}")


Best Poly Ridge:     CV RMSE = 5.786 | Test RMSE = 5.483 | R2 = 0.883
Best Random Forest:  CV RMSE = 5.283 | Test RMSE = 5.356 | R2 = 0.889
Champion GBDT:       CV RMSE = 4.963 | Test RMSE = 4.555 | R2 = 0.919


## 7. Results and Detailed Analysis

### Comprehensive Comparison Table

| Model | Features | Hyperparameters | 5-Fold CV RMSE | Test MAE (MPa) | Test RMSE (MPa) | Test $R^2$ |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: |
| **Linear Regression** | Raw (8) | None (OLS) | $10.627 \pm 0.477$ | 7.746 | 9.796 | 0.628 |
| **Linear Regression** | Engineered (10) | None (OLS) | $7.060 \pm 0.527$ | 5.295 | 6.562 | 0.833 |
| **Polynomial + Ridge** | Raw (8) | degree=2, $\alpha=100.0$ | $7.908 \pm 0.569$ | 5.963 | 7.430 | 0.786 |
| **Polynomial + Ridge** | Engineered (10) | degree=2, $\alpha=10.0$ | $5.786 \pm 0.437$ | 4.358 | 5.483 | 0.883 |
| **Random Forest** | Raw (8) | n_est=200, depth=15 | $5.312 \pm 0.442$ | 3.782 | 5.539 | 0.881 |
| **Random Forest** | Engineered (10) | n_est=200, depth=15 | $5.283 \pm 0.321$ | 3.694 | 5.356 | 0.889 |
| **Gradient Boosting (Best)** | Engineered (10) | n_est=200, lr=0.1, depth=5 | $\mathbf{4.963 \pm 0.353}$ | $\mathbf{2.917}$ | $\mathbf{4.555}$ | $\mathbf{0.919}$ |
| **MLPRegressor** | Engineered (10) | hidden=(64,32), $\alpha=0.01$ | $4.926 \pm 0.571$ | 3.554 | 5.060 | 0.901 |

### Answers to Mandatory Analysis Questions

1. **Why does the linear baseline under-perform? Which assumption of linear regression is violated?**  
   The ordinary linear baseline performs poorly on raw features ($R^2 = 0.628$, RMSE = $9.796\text{ MPa}$) because concrete compressive strength follows non-linear physical kinetics that violate the **linearity assumption of the Gauss-Markov theorem**. Cement hydration exhibits asymptotic growth: strength accelerates steeply during the first 7 to 14 days and subsequently plateaus logarithmically towards 365 days. A straight linear coefficient treats a one-day increment between Day 1 and 2 identically to an increment between Day 100 and 101, leading to severe under-prediction of mature batches and catastrophic over-prediction of green concrete.

2. **Did the engineered features help every model, or only some? Why?**  
   The engineered features ($rac{\text{Water}}{\text{Cement}}$ ratio and $\log(\text{Age})$) dramatically improved linear and polynomial models while offering moderate refinements to tree-based ensembles. For Linear Regression, testing $R^2$ leaped from 0.628 to 0.833 (a 33% reduction in RMSE), and Polynomial Ridge improved from 0.786 to 0.883. This occurs because the logarithmic transformation linearizes the time dimension, allowing a linear hypothesis to model logarithmic hydration directly. In contrast, Random Forest and Gradient Boosting partition feature space via orthogonal axis-aligned step functions, natively approximating non-linearities even without transformations ($R^2 = 0.881$ on raw vs. $0.889$ on engineered for RF). However, providing the explicit ratio $\frac{w}{c}$ still reduces tree depth requirements by consolidating two correlated splits into one master physical feature.

3. **Where are the largest errors (for example very young samples)? What does feature importance tell an engineer?**  
   Inspection of the residual plot demonstrates that the largest absolute prediction residuals ($|e_i| > 10\text{ MPa}$) occur predominantly in two domains: (i) very young specimens (curing age $\le 7$ days) where hydration reactions are highly volatile and sensitive to ambient curing temperatures not captured in the dataset, and (ii) high-strength formulations ($> 65\text{ MPa}$) with high concentrations of superplasticizers and blast-furnace slag, where secondary pozzolanic reactions exhibit complex multi-phase microstructural interlocking. Feature importance analysis confirms that **Curing Age** (and its transform $\log(\text{Age})$), **Cement content**, and **Water/Cement ratio** account for over 72% of predictive split gains. For civil engineers, this reinforces the classic Abrams' rule: minimizing water while maintaining workability via plasticizers is the single most decisive lever for concrete structural performance.


## 8. Conclusion
In this work, we developed an end-to-end regression pipeline for predicting concrete compressive strength. We demonstrated that bridging physical domain formulations (Abrams' law and logarithmic curing kinetics) with modern gradient boosting regression yields an exceptional $R^2$ of 0.919 and RMSE of 4.555 MPa. The baseline linear model failed on raw features due to severe non-linearity, but was revived to $R^2 = 0.833$ via feature engineering. Future directions include incorporating ambient curing temperature telemetry and physics-informed neural networks (PINNs) to constrain predictions within physical bounding limits.


## AI Use Disclosure
I used Antigravity (powered by Google DeepMind Gemini) for assisting with pipeline scaffolding, exploratory data visualization code, and mathematical phrasing of the problem statement. All experiments, cross-validations, metrics, and analyses were produced, verified, and checked by me, and I take full responsibility for this work.


## References
[1] I-C. Yeh, "Modeling of strength of high-performance concrete using artificial neural networks," *Cement and Concrete Research*, vol. 28, no. 12, pp. 1797-1808, 1998.  
[2] F. Pedregosa et al., "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825-2830, 2011.
